In [11]:
import os

from Bio import Entrez
import pandas as pd
import plotly.express as px

from cstarters.ml_utils import load_dataset 

data_dir = "../../data"
out_dir = "../../out"

In [2]:
Entrez.email = "your_email@example.com"

def get_taxonomy(
    taxon_name: str,
    rank: str = "species",
    required_ancestor_taxid: int | None = 2,
    taxid: str | int | None = None,
) -> dict[str, str | None]:

    # If an exception TaxID was provided, bypass the name search.
    if taxid is not None:
        tax_ids = [str(taxid)]

    else:
        query = (
            f'"{taxon_name}"[Scientific Name] '
            f"AND {rank}[Rank]"
        )

        with Entrez.esearch(
            db="taxonomy",
            term=query,
            retmax=100,
        ) as handle:
            tax_ids = Entrez.read(handle)["IdList"]

        if not tax_ids:
            raise ValueError(
                f"No {rank} named {taxon_name!r} found"
            )

    with Entrez.efetch(
        db="taxonomy",
        id=",".join(tax_ids),
        retmode="xml",
    ) as handle:
        records = Entrez.read(handle)

    valid_records = []

    for record in records:
        nodes = list(record["LineageEx"]) + [record]

        lineage_taxids = {
            str(node["TaxId"])
            for node in nodes
        }

        # Only enforce the requested rank when searching by name.
        # A direct TaxID might represent a strain or a "no rank" node.
        if taxid is None and str(record["Rank"]).lower() != rank.lower():
            continue

        if (
            required_ancestor_taxid is not None
            and str(required_ancestor_taxid) not in lineage_taxids
        ):
            continue

        valid_records.append(record)

    if len(valid_records) != 1:
        candidates = [
            (
                str(record["TaxId"]),
                str(record["ScientificName"]),
                str(record["Rank"]),
            )
            for record in records
        ]

        raise ValueError(
            f"Could not uniquely resolve {taxon_name!r}. "
            f"Candidates: {candidates}"
        )

    record = valid_records[0]
    nodes = list(record["LineageEx"]) + [record]

    lineage_by_rank = {
        str(node["Rank"]).lower(): str(node["ScientificName"])
        for node in nodes
    }

    return {
        "query_name": taxon_name,
        "accepted_name": str(record["ScientificName"]),
        "ncbi_taxid": str(record["TaxId"]),
        "species": lineage_by_rank.get("species"),
        "genus": lineage_by_rank.get("genus"),
        "family": lineage_by_rank.get("family"),
        "order": lineage_by_rank.get("order"),
        "class": lineage_by_rank.get("class"),
        "phylum": lineage_by_rank.get("phylum"),
        "kingdom": lineage_by_rank.get("kingdom"),
        "domain": (
            lineage_by_rank.get("domain")
            or lineage_by_rank.get("superkingdom")
        ),
    }


taxonomy = get_taxonomy("Bacillus licheniformis", rank='species')

taxonomy

{'query_name': 'Bacillus licheniformis',
 'accepted_name': 'Bacillus licheniformis',
 'ncbi_taxid': '1402',
 'species': 'Bacillus licheniformis',
 'genus': 'Bacillus',
 'family': 'Bacillaceae',
 'order': 'Caryophanales',
 'class': 'Bacilli',
 'phylum': 'Bacillota',
 'kingdom': 'Bacillati',
 'domain': 'Bacteria'}

In [20]:
df = load_dataset(path = os.path.join(data_dir, "cstarters_dataset.csv"), extra_columns=['ncbi_organism_species'])
df.head()

,cs_accession,starter_beta_hydroxylation,starter_category,aligned_sequence,ncbi_organism_genus,ncbi_organism_species
1,CS0000126,NaN,SCFA,----------------------------------------------...,Streptomyces,Streptomyces platensis
3,CS0000291,FALSE,LCFA,------------------M----L------------E---------...,Streptomyces,Streptomyces fradiae
4,CS0000296,NaN,AR,------------------M----F------------V---------...,Streptomyces,Streptomyces anulatus
5,CS0000305,TRUE,LCFA,------------------P----A------------V----A----...,Pseudomonas,Pseudomonas sp. MIS38
6,CS0000306,FALSE,LCFA,------------------M----S------------A---------...,Streptomyces,Streptomyces filamentosus


In [22]:
taxonomy_df = pd.read_csv("cstarters_dataset_taxonomy.csv")

reset_dataset = False
if reset_dataset:
    taxonomy_df = pd.DataFrame(columns=["cs_accession", "starter_category", "species", "genus", "family", "order", "class", "phylum", "kingdom", "domain"])

In [ ]:
specific_taxid_cases = {
    "Burkholderia sp. FERM BP-3421": "1494466",
    "Stigmatella aurantiaca Sg a15": "675526",
    "Burkholderia cenocepacia H111": "1055524",
    "Planktothrix serta PCC 8927": "671068",
}

existing_accessions = set(taxonomy_df["cs_accession"])

for index, entry in df.iterrows():
    print(entry["cs_accession"], entry["ncbi_organism_species"], entry["starter_category"])
    if entry["cs_accession"] in existing_accessions:
        continue
    if entry["ncbi_organism_species"] not in ["uncultured organism"]:
        preferred_taxid = specific_taxid_cases.get(entry["ncbi_organism_species"])
        entry_taxonomy = get_taxonomy(entry["ncbi_organism_species"], rank="species", taxid=preferred_taxid)
        taxonomy_df.loc[len(taxonomy_df)] = {
            "cs_accession": entry["cs_accession"],
            "starter_category": entry["starter_category"], 
            "species": entry_taxonomy["species"], 
            "genus": entry_taxonomy["genus"], 
            "family": entry_taxonomy["family"], 
            "order": entry_taxonomy["order"], 
            "class": entry_taxonomy["class"], 
            "phylum": entry_taxonomy["phylum"], 
            "kingdom": entry_taxonomy["kingdom"], 
            "domain": entry_taxonomy["domain"]
        }
        print(" -> ".join(name for name in entry_taxonomy.values() if name))
    else:
        taxonomy_df.loc[len(taxonomy_df)] = {
            "cs_accession": entry["cs_accession"],
            "starter_category": entry["starter_category"], 
            "species": entry["ncbi_organism_species"], 
            "genus": "", 
            "family": "", 
            "order": "", 
            "class": "", 
            "phylum": "", 
            "kingdom": "", 
            "domain": ""
        }
taxonomy_df.to_csv("cstarters_dataset_taxonomy.csv", index=False)

CS0000126 Streptomyces platensis SCFA
CS0000291 Streptomyces fradiae LCFA
CS0000296 Streptomyces anulatus AR
CS0000305 Pseudomonas sp. MIS38 LCFA
CS0000306 Streptomyces filamentosus LCFA
CS0000309 Bacillus subtilis AR
CS0000315 Streptomyces coelicolor SCFA
CS0000323 Pseudomonas cichorii LCFA
CS0000336 Streptomyces filamentosus LCFA
CS0000339 Streptomyces lasalocidi AR
CS0000341 Streptomyces fungicidicus LCFA
CS0000343 Pseudomonas sp. J465 AR
CS0000346 Goodfellowiella coeruleoviolacea SCFA
CS0000349 Saccharopolyspora erythraea SCFA
CS0000350 uncultured organism FA
CS0000354 Actinoplanes friuliensis LCFA
CS0000359 Thermobifida fusca AR
CS0000362 Mycobacterium avium LCFA
CS0000363 Mycobacteroides chelonae LCFA
CS0000364 Mycobacteroides abscessus LCFA
CS0000368 Streptomyces sp. ATCC 700974 AR
CS0000369 Anabaena sp. Syke748 LCFA
CS0000371 Rhodococcus erythropolis AR
CS0000374 Streptomyces griseoflavus AR
CS0000379 Streptomyces viridochromogenes LCFA
CS0000381 Bacillus licheniformis LCFA
CS0

In [40]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb


MIN_COUNT = 5
LEVELS = ["phylum", "class", "order",  "genus"]
taxonomy_df = taxonomy_df.dropna()
df = taxonomy_df[LEVELS].copy()

for level in LEVELS:
    df[level] = (
        df[level]
        .fillna("Unknown")
        .replace("", "Unknown")
    )

rank_counts = {
    level: df[level].value_counts()
    for level in LEVELS
}


def is_visible(prefix):
    """Every taxon in this path must meet MIN_COUNT at its rank."""
    return all(
        rank_counts[level].get(name, 0) >= MIN_COUNT
        for level, name in zip(LEVELS, prefix)
    )


def make_id(prefix):
    return "|||".join(prefix)


nodes = []

for depth in range(1, len(LEVELS) + 1):
    prefix_levels = LEVELS[:depth]

    prefix_counts = (
        df.groupby(prefix_levels, sort=False, dropna=False)
        .size()
        .reset_index(name="value")
    )

    for row in prefix_counts.itertuples(index=False, name=None):
        prefix = tuple(row[:-1])
        value = row[-1]

        if not is_visible(prefix):
            continue

        nodes.append({
            "id": make_id(prefix),
            "label": prefix[-1],
            "parent": make_id(prefix[:-1]) if depth > 1 else "",
            "value": value,
            "depth": depth,
            "top_level": prefix[0],
            "is_gap": False,
        })


node_values = {
    node["id"]: node["value"]
    for node in nodes
}

children_total = {}

for node in nodes:
    children_total[node["parent"]] = (
        children_total.get(node["parent"], 0)
        + node["value"]
    )

root_remainder = len(df) - children_total.get("", 0)

if root_remainder > 0:
    nodes.append({
        "id": "__gap_root__",
        "label": "",
        "parent": "",
        "value": root_remainder,
        "depth": 1,
        "top_level": None,
        "is_gap": True,
    })

for parent_id, parent_value in node_values.items():
    remainder = parent_value - children_total.get(parent_id, 0)

    if remainder > 0:
        nodes.append({
            "id": f"__gap__{parent_id}",
            "label": "",
            "parent": parent_id,
            "value": remainder,
            "depth": parent_id.count("|||") + 2,
            "top_level": None,
            "is_gap": True,
        })

In [41]:
def lighten(hex_color, amount):
    r, g, b = hex_to_rgb(hex_color)

    return (
        f"rgb("
        f"{round(r + (255 - r) * amount)},"
        f"{round(g + (255 - g) * amount)},"
        f"{round(b + (255 - b) * amount)}"
        f")"
    )

top_level_names = list(dict.fromkeys(
    node["top_level"]
    for node in nodes
    if node["top_level"] is not None
))

palette = px.colors.qualitative.Plotly

top_colors = {
    name: palette[i % len(palette)]
    for i, name in enumerate(top_level_names)
}

colors = []

for node in nodes:
    if node["is_gap"]:
        colors.append("white")
    else:
        base = top_colors[node["top_level"]]
        colors.append(lighten(base, 0.10 * (node["depth"] - 1)))

fig = go.Figure(go.Sunburst(
    ids=[node["id"] for node in nodes],
    labels=[node["label"] for node in nodes],
    parents=[node["parent"] for node in nodes],
    values=[node["value"] for node in nodes],
    branchvalues="total",
    marker=dict(
        colors=colors,
        line=dict(color="white", width=1),
    ),
    insidetextorientation="auto",
    sort=False,
    hovertemplate="%{label}<br>Count: %{value}<extra></extra>",
))
fig.update_traces(textfont_color="black")
fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0),
    paper_bgcolor="white",
)

fig.show()

In [44]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

groups = {
    "Pseudomonas":        taxonomy_df[taxonomy_df["genus"] == "Pseudomonas"],
    "Streptomyces":       taxonomy_df[taxonomy_df["genus"] == "Streptomyces"],
    "Bacilli":            taxonomy_df[taxonomy_df["class"] == "Bacilli"],
    "Betaproteobacteria": taxonomy_df[taxonomy_df["class"] == "Betaproteobacteria"],
}

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "pie"}] * 2] * 2,
    subplot_titles=list(groups.keys()),
)

for (row, col), (name, group_df) in zip([(1,1),(1,2),(2,1),(2,2)], groups.items()):
    counts = group_df["starter_category"].value_counts()
    fig.add_trace(
        go.Pie(labels=counts.index, values=counts.values, name=name,
               textfont=dict(color="black", size=16)),
        row=row, col=col,
    )

fig.update_layout(
    margin=dict(t=40, l=0, r=0, b=0),
    font=dict(size=16),
)
fig.show()